In [7]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import MinMaxScaler

print("=== 📊 شروع فاز ۱: تحلیل اکتشافی داده‌ها (EDA) ===")

# ۱. ساخت ساختار پوشه‌ها برای گیت‌هاب
os.makedirs('reports/figures', exist_ok=True)

# ۲. بارگذاری و بازسازی سریع ساختار مستر از روی فایل اصلی شما
df = pd.read_csv('C:\\Windows\\System32\\protein-recommendation-ai\\protein-recommendation-ai\\data\\interim\\NHANES_Merged_Data.csv')
df_clean = df[df['RIDAGEYR'] >= 18].copy()

# پر کردن مقادیر مفقود با میانه برای حفظ پایداری توزیع آماری
for col in ['DXDTOLE', 'DXDTOPF', 'BMXWT', 'BMXHT', 'BMXBMI', 'LBDHDD', 'LBXTR']:
    df_clean[col] = df_clean[col].fillna(df_clean[col].median())

# ۳. اعمال ویژگی‌های مهندسی‌شده
df_clean['Lean_Mass_kg'] = (df_clean['DXDTOLE'] / 1000).round(2)
df_clean['Body_Fat_Percent'] = df_clean['DXDTOPF'].round(2)
df_clean['Daily_Protein_Intake_g'] = df_clean[['DR1TPROT', 'DR2TPROT']].mean(axis=1)
df_clean['Daily_Protein_Intake_g'] = df_clean['Daily_Protein_Intake_g'].fillna(df_clean['Daily_Protein_Intake_g'].median()).round(2)

scaler = MinMaxScaler(feature_range=(0, 100))
df_clean['Raw_Genetic_Score'] = (df_clean['DXDTOLE'] / df_clean['DXDTOLE'].median()) + \
                                (df_clean['LBDHDD'] / df_clean['LBDHDD'].median()) - \
                                (df_clean['LBXTR'] / df_clean['LBXTR'].median())
df_clean['Metabolic_Profile_Score'] = scaler.fit_transform(df_clean[['Raw_Genetic_Score']]).round(2)

df_clean['Vigorous_Activity'] = df_clean['PAQ650'].apply(lambda x: 1 if x == 1 else 0)
df_clean['Moderate_Activity'] = df_clean['PAQ665'].apply(lambda x: 1 if x == 1 else 0)
df_clean['Activity_Score'] = df_clean['Vigorous_Activity'] * 2 + df_clean['Moderate_Activity'] + 1

# محاسبه تارگت
df_clean['Protein_Requirement_g'] = (df_clean['Lean_Mass_kg'] * 1.2) + (df_clean['BMXWT'] * (df_clean['Activity_Score'] * 0.1)) + (df_clean['Metabolic_Profile_Score'] / 10)
df_clean['Protein_Requirement_g'] = df_clean['Protein_Requirement_g'].round(2)

final_cols = {
    'RIDAGEYR': 'Age', 'BMXWT': 'Weight_kg', 'BMXHT': 'Height_cm', 'BMXBMI': 'BMI',
    'Body_Fat_Percent': 'Body_Fat_Percent', 'Lean_Mass_kg': 'Lean_Mass_kg',
    'Activity_Score': 'Activity_Score', 'Daily_Protein_Intake_g': 'Daily_Protein_Intake_g',
    'Metabolic_Profile_Score': 'Metabolic_Profile_Score', 'Protein_Requirement_g': 'Protein_Requirement_g'
}
df_master = df_clean[list(final_cols.keys())].rename(columns=final_cols)

# تنظیم استایل پیشرفته برای پلات‌ها
sns.set_theme(style="whitegrid")
plt.rcParams.update({'font.size': 10, 'axes.labelsize': 11, 'axes.titlesize': 12})

# -------------------------------------------------------------
# 📊 نمودار اول: توزیع چگالی و فراوانی متغیر هدف (Target Distribution)
# -------------------------------------------------------------
plt.figure(figsize=(6, 4))
sns.histplot(df_master['Protein_Requirement_g'], kde=True, color='#2bc0a4', bins=30, edgecolor='black', alpha=0.7)
plt.title('Distribution of Target: Protein Requirement (g)', fontweight='bold', pad=15)
plt.xlabel('Protein Requirement (g)')
plt.ylabel('Count')
plt.tight_layout()
plt.savefig('C:\\Windows\\System32\\protein-recommendation-ai\\protein-recommendation-ai\\reports\\figures\\01_target_distribution.png', dpi=300)
plt.close()
print("[✓] نمودار اول ذخیره شد: 01_target_distribution.png")

# -------------------------------------------------------------
# 📊 نمودار دوم: مقایسه وزن در برابر تارگت به تفکیک سطح فعالیت (Scatter Plot)
# -------------------------------------------------------------
plt.figure(figsize=(7, 5))
scatter = plt.scatter(
    df_master['Weight_kg'], 
    df_master['Protein_Requirement_g'], 
    c=df_master['Activity_Score'], 
    cmap='viridis', 
    alpha=0.6, 
    edgecolors='none',
    s=25
)
cbar = plt.colorbar(scatter)
cbar.set_label('Physical Activity Score (1 to 4)')
plt.title('Physiological Baseline: Weight vs. Protein Requirement', fontweight='bold', pad=15)
plt.xlabel('Total Body Weight (kg)')
plt.ylabel('Protein Requirement (g)')
plt.grid(True, linestyle='--', alpha=0.5)
plt.tight_layout()
plt.savefig('C:\\Windows\\System32\\protein-recommendation-ai\\protein-recommendation-ai\\reports\\figures\\02_weight_vs_protein.png', dpi=300)
plt.close()
print("[✓] نمودار دوم ذخیره شد: 02_weight_vs_protein.png")

# -------------------------------------------------------------
# 📊 نمودار سوم: ماتریس همبستگی پیرسون ویژگی‌ها (Correlation Matrix Heatmap)
# -------------------------------------------------------------
plt.figure(figsize=(9, 7))
corr_matrix = df_master.corr()
mask = np.triu(np.ones_like(corr_matrix, dtype=bool)) # ماسک کردن برای زیبایی و عدم تکرار داده‌ها

sns.heatmap(
    corr_matrix, 
    mask=mask, 
    annot=True, 
    fmt=".2f", 
    cmap='coolwarm', 
    vmax=1.0, 
    vmin=-1.0, 
    square=True, 
    linewidths=0.5,
    cbar_kws={"shrink": 0.8}
)
plt.title('Feature Correlation Matrix (Pearson Coefficients)', fontweight='bold', pad=20)
plt.tight_layout()
plt.savefig('C:\\Windows\\System32\\protein-recommendation-ai\\protein-recommendation-ai\\reports\\figures\\03_correlation_matrix.png', dpi=300)
plt.close()
print("[✓] نمودار سوم ذخیره شد: 03_correlation_matrix.png")

print("\n🎉 تمام تصاویر با موفقیت در پوشه ی reports/figures/ ذخیره شدند و آماده آپلود در گیت‌هاب هستند.")

=== 📊 شروع فاز ۱: تحلیل اکتشافی داده‌ها (EDA) ===
[✓] نمودار اول ذخیره شد: 01_target_distribution.png
[✓] نمودار دوم ذخیره شد: 02_weight_vs_protein.png
[✓] نمودار سوم ذخیره شد: 03_correlation_matrix.png

🎉 تمام تصاویر با موفقیت در پوشه ی reports/figures/ ذخیره شدند و آماده آپلود در گیت‌هاب هستند.
